# OULAD Data Quality: Bri's `01-raw` build

**This notebook reads `` `ftw-week-07`.`01-raw` `` which is the original ingestion.**
For the rebuild, use `ina_dq_raw` instead.

Results are written to the shared `` `01-raw-dev`.dq_check_results `` table under
`layer = 'raw-orig'`, so both builds sit in one table and can be compared
directly. 

Identical check logic to the dev notebook. Only the source schema and the layer
label differ, so any difference in results is a difference in the data.

**Run order:** cells 1–2, then the dataset blocks, then `row_count_not_empty`,
then VOLUME. The comparison query is the last cell, run the dev notebook first
so there is something to compare against.


## 1. Results store and run context

Each run gets a `run_id`. Re-running one dataset cell deletes only that cell's rows *for the current run*, so cells stay idempotent while the trend survives.

The variable is `dq_run_id`; the column is `run_id`. Different names on purpose, an unqualified reference matching a column name resolves to the column and silently turns the filter into a no-op.

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 0. RESULTS STORE
-- ---------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS `ftw-week-07`.`01-raw-dev`.dq_check_results (
    run_id        STRING,
    executed_at   TIMESTAMP,
    layer         STRING,     -- raw | clean | mart
    dataset       STRING,
    check_name    STRING,
    check_type    STRING,     -- NOT_NULL UNIQUE RANGE DOMAIN REFERENTIAL VOLUME FORMAT CONSISTENCY RECONCILIATION MEASURE
    status        STRING,     -- PASS | WARN | FAIL | INFO
    severity      STRING,     -- FAIL | WARN | INFO
    fail_count    BIGINT,
    total_count   BIGINT,
    fail_pct      DOUBLE,
    threshold_pct DOUBLE,
    metric_value  DOUBLE,     -- for measurements; never a failure count
    owner         STRING,
    details       STRING
)
COMMENT 'One row per check per run. Append only.';

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1. RUN CONTEXT
--    Variable is dq_run_id; the column is run_id. Different names on
--    purpose — an unqualified reference to a name that matches a column
--    resolves to the column and silently breaks the filter.
-- ---------------------------------------------------------------------
DECLARE OR REPLACE VARIABLE dq_run_id STRING;
SET VARIABLE dq_run_id = uuid();

## 2. Check blocks

Pattern per dataset: one `total` CTE, one `checks` CTE listing every check as a row, one SELECT that turns `fail_count` + `threshold_pct` + `severity` into a status. Adding a check means adding a `UNION ALL` branch, not another INSERT.

`TRY_CAST` is mandatory on every numeric comparison. raw is all STRING: a bare `col <= 0` casts implicitly, turns `?` into NULL, and `NULL <= 0` is NULL, the row is never counted and the check passes when it should fail.

In [0]:
%sql
-- ========== courses ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'courses';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.courses
),
checks AS (
    SELECT 'unique_module_presentation' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT code_module, code_presentation
                FROM `ftw-week-07`.`01-raw`.courses
                GROUP BY code_module, code_presentation HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK fans out every downstream join.' AS details
    UNION ALL
    SELECT 'not_null_all_columns', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.courses
                 WHERE code_module IS NULL OR code_presentation IS NULL
                    OR module_presentation_length IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'All three columns are structural. Nulls break joins and duration logic.'
    UNION ALL
    -- TRY_CAST is required now that raw is all STRING. A bare `col <= 0`
    -- casts implicitly, turns a sentinel into NULL, and NULL <= 0 is NULL,
    -- so the row is never counted and the check passes when it should not.
    SELECT 'range_length_positive', 'RANGE', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.courses
                 WHERE TRY_CAST(module_presentation_length AS INT) IS NULL
                    OR TRY_CAST(module_presentation_length AS INT) <= 0) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Non-positive or unparseable duration. A course cannot last zero days.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'courses',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- ========== assessments ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'assessments';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.assessments
),
checks AS (
    SELECT 'unique_id_assessment' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT id_assessment FROM `ftw-week-07`.`01-raw`.assessments
                GROUP BY id_assessment HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK double-counts every submission.' AS details
    UNION ALL
    SELECT 'not_null_structural', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE id_assessment IS NULL OR code_module IS NULL OR code_presentation IS NULL
                    OR assessment_type IS NULL OR weight IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           '`date` excluded — Exam rows legitimately lack one.'
    UNION ALL
    SELECT 'fk_to_courses', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments a
                 LEFT JOIN `ftw-week-07`.`01-raw`.courses c
                        ON a.code_module = c.code_module
                       AND a.code_presentation = c.code_presentation
                 WHERE c.code_module IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Assessment with no parent course presentation.'
    UNION ALL
    SELECT 'domain_assessment_type', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE assessment_type IS NULL
                    OR assessment_type NOT IN ('TMA', 'CMA', 'Exam')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Expected TMA, CMA, or Exam.'
    UNION ALL
    SELECT 'range_weight_0_100', 'RANGE', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE TRY_CAST(weight AS DOUBLE) IS NULL
                    OR TRY_CAST(weight AS DOUBLE) < 0
                    OR TRY_CAST(weight AS DOUBLE) > 100) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Weight is a percentage. Unparseable counts as a failure.'
    UNION ALL
    SELECT 'format_date_numeric', 'FORMAT', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE `date` IS NOT NULL AND TRIM(`date`) NOT IN ('', '?')
                   AND TRY_CAST(`date` AS INT) IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Present-but-unparseable day offsets would silently null out in Clean.'
    UNION ALL
    SELECT 'missing_date_only_on_exams', 'CONSISTENCY', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE (`date` IS NULL OR TRIM(`date`) IN ('', '?'))
                   AND assessment_type <> 'Exam') AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Only Exam rows may lack a due day. A TMA/CMA without one is a real gap.'
    UNION ALL
    -- Business rule, checked at raw because Mart cannot compute a weighted
    -- final score without it. Known violations: CCC has two Exam rows per
    -- presentation at weight 100 each, and GGG has TMA weights summing to 0.
    SELECT 'weight_sums_per_presentation', 'CONSISTENCY', 'WARN', 0.0,
           CAST((SELECT COUNT(*) FROM (
                SELECT code_module, code_presentation,
                       SUM(CASE WHEN assessment_type IN ('TMA','CMA')
                                THEN TRY_CAST(weight AS DOUBLE) ELSE 0 END) AS continuous_weight,
                       SUM(CASE WHEN assessment_type = 'Exam'
                                THEN TRY_CAST(weight AS DOUBLE) ELSE 0 END) AS exam_weight
                FROM `ftw-week-07`.`01-raw`.assessments
                GROUP BY code_module, code_presentation
                HAVING continuous_weight <> 100 OR exam_weight <> 100)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Presentations where TMA+CMA <> 100 or Exam <> 100. Blocks any weighted-score measure.'
    UNION ALL
    SELECT 'measure_missing_due_date', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
                 WHERE `date` IS NULL OR TRIM(`date`) IN ('', '?')) AS DOUBLE),
           'Assessments with no due day. Expected: Exam rows only.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'assessments',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
-- ========== vle ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'vle';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.vle
),
checks AS (
    SELECT 'unique_id_site' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT id_site FROM `ftw-week-07`.`01-raw`.vle
                GROUP BY id_site HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK double-counts clicks when student_vle joins in.' AS details
    UNION ALL
    SELECT 'not_null_structural', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
                 WHERE id_site IS NULL OR code_module IS NULL
                    OR code_presentation IS NULL OR activity_type IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'week_from/week_to excluded — absent for always-on resources.'
    UNION ALL
    SELECT 'fk_to_courses', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle v
                 LEFT JOIN `ftw-week-07`.`01-raw`.courses c
                        ON v.code_module = c.code_module
                       AND v.code_presentation = c.code_presentation
                 WHERE c.code_module IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Resource with no parent course presentation.'
    UNION ALL
    -- ACCEPTED VALUES, per slide 38. WARN not FAIL: a new activity type is
    -- news about the source, not corruption of what is already loaded.
    SELECT 'domain_activity_type', 'DOMAIN', 'WARN', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
                 WHERE activity_type NOT IN (
                    'dataplus','dualpane','externalquiz','folder','forumng','glossary',
                    'homepage','htmlactivity','oucollaborate','oucontent','ouelluminate',
                    'ouwiki','page','questionnaire','quiz','repeatactivity','resource',
                    'sharedsubpage','subpage','url')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Activity type outside the 20 known values. New type = update the list and the dictionary.'
    UNION ALL
    SELECT 'format_week_range_numeric', 'FORMAT', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
                 WHERE (week_from IS NOT NULL AND TRIM(week_from) NOT IN ('', '?')
                        AND TRY_CAST(week_from AS INT) IS NULL)
                    OR (week_to   IS NOT NULL AND TRIM(week_to)   NOT IN ('', '?')
                        AND TRY_CAST(week_to   AS INT) IS NULL)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Present-but-unparseable week bounds.'
    UNION ALL
    SELECT 'consistency_week_from_le_week_to', 'CONSISTENCY', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
                 WHERE TRY_CAST(week_from AS INT) IS NOT NULL
                   AND TRY_CAST(week_to   AS INT) IS NOT NULL
                   AND TRY_CAST(week_from AS INT) > TRY_CAST(week_to AS INT)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'A resource cannot close before it opens.'
    UNION ALL
    SELECT 'measure_no_week_range', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
                 WHERE week_from IS NULL OR TRIM(week_from) IN ('', '?')) AS DOUBLE),
           'Resources with no week range. Expected for always-on resources.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'vle',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
-- ========== student_info ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'student_info';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.student_info
),
checks AS (
    SELECT 'unique_enrollment_grain' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT code_module, code_presentation, id_student
                FROM `ftw-week-07`.`01-raw`.student_info
                GROUP BY code_module, code_presentation, id_student
                HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Grain is the enrollment triplet, not id_student. Duplicates = conflicting outcomes.' AS details
    UNION ALL
    SELECT 'not_null_structural', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE code_module IS NULL OR code_presentation IS NULL OR id_student IS NULL
                    OR gender IS NULL OR final_result IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'imd_band excluded — documented as legitimately missing.'
    UNION ALL
    SELECT 'fk_to_courses', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info si
                 LEFT JOIN `ftw-week-07`.`01-raw`.courses c
                        ON si.code_module = c.code_module
                       AND si.code_presentation = c.code_presentation
                 WHERE c.code_module IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Enrollment with no parent course presentation.'
    UNION ALL
    SELECT 'domain_gender', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE gender NOT IN ('M', 'F')) AS BIGINT),
           CAST(NULL AS DOUBLE), 'Expected M or F.'
    UNION ALL
    SELECT 'domain_final_result', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE final_result NOT IN ('Withdrawn','Fail','Pass','Distinction')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Drives every dropout question. Drift here corrupts the dashboard silently.'
    UNION ALL
    SELECT 'domain_disability', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE disability NOT IN ('Y', 'N')) AS BIGINT),
           CAST(NULL AS DOUBLE), 'Expected Y or N.'
    UNION ALL
    SELECT 'domain_age_band', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE age_band NOT IN ('0-35','35-55','55<=')) AS BIGINT),
           CAST(NULL AS DOUBLE), 'Expected one of three bands.'
    UNION ALL
    SELECT 'domain_highest_education', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE highest_education NOT IN (
                    'A Level or Equivalent','HE Qualification','Lower Than A Level',
                    'No Formal quals','Post Graduate Qualification')) AS BIGINT),
           CAST(NULL AS DOUBLE), 'Expected one of five levels.'
    UNION ALL
    -- Source inconsistency: nine bands carry a % suffix, '10-20' does not.
    -- Left as WARN because it is a labelling defect, not a missing value —
    -- Clean normalizes it, and this check is what proves Clean did.
    SELECT 'domain_imd_band_format', 'DOMAIN', 'WARN', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE imd_band NOT IN (
                    '0-10%','10-20%','20-30%','30-40%','40-50%',
                    '50-60%','60-70%','70-80%','80-90%','90-100%','?')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Bands outside the canonical set. `10-20` ships without the % suffix in the source.'
    UNION ALL
    SELECT 'range_attempts_credits', 'RANGE', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE TRY_CAST(num_of_prev_attempts AS INT) IS NULL
                    OR TRY_CAST(num_of_prev_attempts AS INT) < 0
                    OR TRY_CAST(studied_credits AS INT) IS NULL
                    OR TRY_CAST(studied_credits AS INT) <= 0) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Negative attempts or non-positive credits are impossible.'
    UNION ALL
    SELECT 'sentinel_in_required_categoricals', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE TRIM(gender) IN ('', '?') OR TRIM(region) IN ('', '?')
                    OR TRIM(age_band) IN ('', '?') OR TRIM(highest_education) IN ('', '?')
                    OR TRIM(disability) IN ('', '?') OR TRIM(final_result) IN ('', '?')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Categoricals are never cast, so nothing else would catch a sentinel here.'
    UNION ALL
    SELECT 'measure_missing_imd_band', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
                 WHERE imd_band IS NULL OR TRIM(imd_band) IN ('', '?')) AS DOUBLE),
           'Enrollments with no IMD band. Clean must store NULL or "?" becomes a category.'
    UNION ALL
    SELECT 'measure_distinct_students', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(DISTINCT id_student)
                 FROM `ftw-week-07`.`01-raw`.student_info) AS DOUBLE),
           'Distinct students vs enrollments. The gap is students taking multiple presentations.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'student_info',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
13,13


In [0]:
%sql
-- ========== student_registration ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'student_registration';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.student_registration
),
checks AS (
    SELECT 'unique_enrollment_grain' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT code_module, code_presentation, id_student
                FROM `ftw-week-07`.`01-raw`.student_registration
                GROUP BY code_module, code_presentation, id_student
                HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicates = two conflicting registrations for one enrollment.' AS details
    UNION ALL
    SELECT 'not_null_keys', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE code_module IS NULL OR code_presentation IS NULL
                    OR id_student IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Key columns only. Both date columns handled separately.'
    UNION ALL
    SELECT 'fk_to_student_info', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration sr
                 LEFT JOIN `ftw-week-07`.`01-raw`.student_info si
                        ON sr.code_module = si.code_module
                       AND sr.code_presentation = si.code_presentation
                       AND sr.id_student = si.id_student
                 WHERE si.id_student IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Registration with no matching enrollment.'
    UNION ALL
    SELECT 'format_dates_numeric', 'FORMAT', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE (date_registration IS NOT NULL AND TRIM(date_registration) NOT IN ('', '?')
                        AND TRY_CAST(date_registration AS INT) IS NULL)
                    OR (date_unregistration IS NOT NULL AND TRIM(date_unregistration) NOT IN ('', '?')
                        AND TRY_CAST(date_unregistration AS INT) IS NULL)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Negative values are valid (registered before module start) and not flagged.'
    UNION ALL
    SELECT 'consistency_unreg_after_reg', 'CONSISTENCY', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE TRY_CAST(date_registration   AS INT) IS NOT NULL
                   AND TRY_CAST(date_unregistration AS INT) IS NOT NULL
                   AND TRY_CAST(date_unregistration AS INT)
                     < TRY_CAST(date_registration   AS INT)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Cannot unregister before registering. Any hit makes duration metrics negative.'
    UNION ALL
    -- Threshold is a share, not a literal. 45/32,593 = 0.14%, well under 0.5%.
    SELECT 'completeness_date_registration', 'NOT_NULL', 'FAIL', 0.005,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE date_registration IS NULL
                    OR TRIM(date_registration) IN ('', '?')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Known gap — see decisions.md. Clean stores NULL, not 0. FAILs above 0.5% of rows.'
    UNION ALL
    SELECT 'completeness_both_dates_missing', 'NOT_NULL', 'FAIL', 0.001,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE (date_registration   IS NULL OR TRIM(date_registration)   IN ('', '?'))
                   AND (date_unregistration IS NULL OR TRIM(date_unregistration) IN ('', '?'))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'No derivable timeline. Flag in Clean so Mart excludes them from duration metrics.'
    UNION ALL
    SELECT 'measure_never_unregistered', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
                 WHERE date_unregistration IS NULL
                    OR TRIM(date_unregistration) IN ('', '?')) AS DOUBLE),
           'Not a gap — this is the non-withdrawal population. Cross-check vs final_result.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'student_registration',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
-- ========== student_assessment ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'student_assessment';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.student_assessment
),
checks AS (
    SELECT 'unique_submission_grain' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (
                SELECT id_assessment, id_student
                FROM `ftw-week-07`.`01-raw`.student_assessment
                GROUP BY id_assessment, id_student HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicates = two conflicting scores for one submission.' AS details
    UNION ALL
    SELECT 'not_null_structural', 'NOT_NULL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
                 WHERE id_assessment IS NULL OR id_student IS NULL
                    OR date_submitted IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'score excluded — absent for non-submissions.'
    UNION ALL
    SELECT 'fk_to_assessments', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment sa
                 LEFT JOIN `ftw-week-07`.`01-raw`.assessments a
                        ON sa.id_assessment = a.id_assessment
                 WHERE a.id_assessment IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Submission against a non-existent assessment.'
    UNION ALL
    -- This table has no module/presentation column, so id_student alone is
    -- the only join available at raw. Clean conforms it via assessments.
    SELECT 'fk_to_student', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (
                SELECT DISTINCT sa.id_student
                FROM `ftw-week-07`.`01-raw`.student_assessment sa
                LEFT ANTI JOIN `ftw-week-07`.`01-raw`.student_info si
                            ON sa.id_student = si.id_student)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Submission from a student with no enrollment record anywhere.'
    UNION ALL
    SELECT 'domain_is_banked', 'DOMAIN', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
                 WHERE is_banked NOT IN ('0', '1')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Expected 0 or 1. Compared as strings — raw is untyped.'
    UNION ALL
    SELECT 'format_range_score', 'FORMAT', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
                 WHERE score IS NOT NULL AND TRIM(score) NOT IN ('', '?')
                   AND (TRY_CAST(score AS DOUBLE) IS NULL
                        OR TRY_CAST(score AS DOUBLE) < 0
                        OR TRY_CAST(score AS DOUBLE) > 100)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Present scores must parse as a number 0-100.'
    UNION ALL
    SELECT 'measure_missing_score', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
                 WHERE score IS NULL OR TRIM(score) IN ('', '?')) AS DOUBLE),
           'Submissions with no score. Clean must use NULL, not 0 — different facts.'
    UNION ALL
    -- Banked scores were carried over from a previous presentation. Any
    -- "average score by presentation" measure that includes them is wrong.
    SELECT 'measure_banked_scores', 'MEASURE', 'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
                 WHERE is_banked = '1') AS DOUBLE),
           'Scores transferred from an earlier presentation, not earned in this one.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'student_assessment',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
-- ========== student_vle ==========
-- 10.6M rows. The row-level checks are folded into ONE pass with conditional aggregation rather than one scan per check. The two FK checks need joins and stay separate. This is the only table where the difference is worth the extra complexity.
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig' AND dataset = 'student_vle';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH agg AS (
    SELECT
        COUNT(*)                                                          AS n,
        SUM(CASE WHEN code_module IS NULL OR code_presentation IS NULL
                   OR id_student IS NULL OR id_site IS NULL
                   OR `date` IS NULL OR sum_click IS NULL
                 THEN 1 ELSE 0 END)                                       AS f_notnull,
        SUM(CASE WHEN TRY_CAST(sum_click AS INT) IS NULL
                   OR TRY_CAST(sum_click AS INT) <= 0
                 THEN 1 ELSE 0 END)                                       AS f_clicks,
        SUM(CASE WHEN `date` IS NOT NULL AND TRIM(`date`) NOT IN ('', '?')
                  AND TRY_CAST(`date` AS INT) IS NULL
                 THEN 1 ELSE 0 END)                                       AS f_date_format,
        SUM(TRY_CAST(sum_click AS BIGINT))                                AS total_clicks,
        COUNT(*) / COUNT(DISTINCT CONCAT_WS('|', code_module, code_presentation,
                                            id_student, id_site, `date`)) AS fanout_ratio
    FROM `ftw-week-07`.`01-raw`.student_vle
),
checks AS (
    -- No UNIQUE check: duplicates on the daily key are EXPECTED at raw.
    -- Clean aggregates them, so what matters is that the GROUP BY key is complete.
    SELECT 'aggregation_key_complete' AS check_name, 'NOT_NULL' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST(a.f_notnull AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'raw is per-session; Clean aggregates. The GROUP BY key must be complete.' AS details
    FROM agg a
    UNION ALL
    SELECT 'range_sum_click_positive', 'RANGE', 'FAIL', 0.0,
           CAST(a.f_clicks AS BIGINT), CAST(NULL AS DOUBLE),
           'A logged interaction implies at least one click.'
    FROM agg a
    UNION ALL
    SELECT 'format_date_numeric', 'FORMAT', 'FAIL', 0.0,
           CAST(a.f_date_format AS BIGINT), CAST(NULL AS DOUBLE),
           'Present-but-unparseable day offsets would silently null out in Clean.'
    FROM agg a
    UNION ALL
    -- Measurement, not a failure. Recorded as metric_value so the dashboard does not read 39.6 million as 39.6 million failed rows.
    SELECT 'reconciliation_total_clicks', 'RECONCILIATION', 'INFO', 0.0,
           CAST(0 AS BIGINT), CAST(a.total_clicks AS DOUBLE),
           'Total clicks at raw. Clean must reproduce this exactly after aggregating.'
    FROM agg a
    UNION ALL
    SELECT 'fanout_ratio_stable', 'RANGE', 'WARN', 0.0,
           CAST(CASE WHEN a.fanout_ratio BETWEEN 1.0 AND 2.0 THEN 0 ELSE 1 END AS BIGINT),
           CAST(a.fanout_ratio AS DOUBLE),
           'Rows per daily key. Drift outside 1.0-2.0 means the source grain changed.'
    FROM agg a
    UNION ALL
    SELECT 'fk_to_vle', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_vle sv
                 LEFT JOIN `ftw-week-07`.`01-raw`.vle v ON sv.id_site = v.id_site
                 WHERE v.id_site IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Interaction with a non-existent VLE resource.'
    UNION ALL
    SELECT 'fk_to_student_info', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_vle sv
                 LEFT JOIN `ftw-week-07`.`01-raw`.student_info si
                        ON sv.code_module = si.code_module
                       AND sv.code_presentation = si.code_presentation
                       AND sv.id_student = si.id_student
                 WHERE si.id_student IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Engagement data for an enrollment that does not exist.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'student_vle',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN a.n > 0 AND c.fail_count / a.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, a.n,
       CASE WHEN a.n = 0 THEN NULL ELSE c.fail_count / a.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN agg a;

num_affected_rows
0


num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
-- ========== cross-table consistency ==========
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig'
  AND check_name = 'row_count_not_empty';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`01-raw`.student_info
),
checks AS (
    -- LEFT JOIN, not INNER. The key sets match today; an inner join would
    -- silently shrink the denominator the day they stop matching.
    SELECT 'withdrawal_flag_agrees_with_date' AS check_name, 'CONSISTENCY' AS check_type,
           'WARN' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*)
                 FROM `ftw-week-07`.`01-raw`.student_info si
                 LEFT JOIN `ftw-week-07`.`01-raw`.student_registration sr
                        ON si.code_module = sr.code_module
                       AND si.code_presentation = sr.code_presentation
                       AND si.id_student = sr.id_student
                 WHERE (si.final_result = 'Withdrawn')
                    <> (sr.date_unregistration IS NOT NULL
                        AND TRIM(sr.date_unregistration) NOT IN ('', '?'))) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'final_result and unregistration date disagree. Pick one as authoritative before Mart.' AS details
    UNION ALL
    SELECT 'enrollment_keys_match_registration', 'REFERENTIAL', 'FAIL', 0.0,
           CAST((SELECT COUNT(*)
                 FROM `ftw-week-07`.`01-raw`.student_info si
                 LEFT ANTI JOIN `ftw-week-07`.`01-raw`.student_registration sr
                            ON si.code_module = sr.code_module
                           AND si.code_presentation = sr.code_presentation
                           AND si.id_student = sr.id_student) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Enrollments with no registration row. Both sides must cover the same key set.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', 'cross_table',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig'
  AND check_name = 'row_count_not_empty';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', dataset,
       'row_count_not_empty', 'VOLUME',
       CASE WHEN n = 0 THEN 'FAIL' ELSE 'PASS' END, 'FAIL',
       CASE WHEN n = 0 THEN 1 ELSE 0 END, n,
       CASE WHEN n = 0 THEN 1.0 ELSE 0.0 END, 0.0,
       CAST(NULL AS DOUBLE), 'data-engineering',
       'Zero rows is a silent failure. Every other check passes vacuously on an empty table.'
FROM (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'raw-orig' AND check_type <> 'VOLUME'
    GROUP BY dataset
);

num_affected_rows
0


num_affected_rows,num_inserted_rows
8,8


## 3. VOLUME

The pipeline reports SUCCESS while the row count collapses. *Not zero* doesn't catch that. This compares each dataset to its own most recent previous run and flags a swing past 10%. It self-baselines, no literal to maintain, and it only works because history is retained.

First run has nothing to compare against; those rows record as `INFO`, not a false PASS.

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 3. VOLUME
--    The pipeline reports SUCCESS while the row count collapses.
--    "Not zero" does not catch that. This compares each dataset's current
--    row count to the most recent PREVIOUS run and flags a swing beyond
--    10%. It self-baselines, no literal to update as data grows.
--
--    Run this AFTER section 2, which is what writes the current counts.
--    On the very first run there is nothing to compare to; those rows
--    record as INFO rather than a false PASS.
-- ---------------------------------------------------------------------
DELETE FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'raw-orig'
  AND check_name = 'volume_stable_vs_previous_run';

INSERT INTO `ftw-week-07`.`01-raw-dev`.dq_check_results
WITH current_counts AS (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'raw-orig' AND check_type <> 'VOLUME'
    GROUP BY dataset
),
previous_counts AS (
    SELECT dataset, total_count AS prev_n
    FROM (
        SELECT dataset, total_count,
               DENSE_RANK() OVER (PARTITION BY dataset ORDER BY executed_at DESC) AS rnk
        FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
        WHERE layer = 'raw-orig' AND check_type <> 'VOLUME' AND run_id <> dq_run_id
    )
    WHERE rnk = 1
    GROUP BY dataset, total_count
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'raw-orig', c.dataset,
       'volume_stable_vs_previous_run', 'VOLUME',
       CASE WHEN p.prev_n IS NULL THEN 'INFO'
            WHEN ABS(c.n - p.prev_n) / p.prev_n <= 0.10 THEN 'PASS'
            ELSE 'WARN' END,
       'WARN',
       CASE WHEN p.prev_n IS NULL OR ABS(c.n - p.prev_n) / p.prev_n <= 0.10
            THEN 0 ELSE ABS(c.n - p.prev_n) END,
       c.n,
       CASE WHEN p.prev_n IS NULL THEN NULL ELSE ABS(c.n - p.prev_n) / p.prev_n END,
       0.10,
       CAST(p.prev_n AS DOUBLE),
       'data-engineering',
       CASE WHEN p.prev_n IS NULL
            THEN 'First run for this dataset. No baseline to compare against yet.'
            ELSE 'Row count vs previous run. metric_value holds the previous count.' END
FROM current_counts c
LEFT JOIN previous_counts p ON c.dataset = p.dataset;

num_affected_rows
0


num_affected_rows,num_inserted_rows
8,8


## 4. raw exit gate

**Clean does not start until this returns zero rows.**

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 4. raw EXIT GATE
--    Clean does not start until this returns zero rows.
-- ---------------------------------------------------------------------
SELECT dataset, check_name, check_type, fail_count, total_count,
       ROUND(fail_pct * 100, 4) AS fail_pct, details
FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
WHERE run_id = dq_run_id AND status = 'FAIL'
ORDER BY dataset, check_type;

dataset,check_name,check_type,fail_count,total_count,fail_pct,details


## 5. DQ dashboard

Data health in 10 seconds. Each tile is its own cell so it can be pinned to a Databricks dashboard individually. All tiles read `v_dq_latest`, which is the most recent run per layer, current state on the dashboard, full history in the table.

In [0]:
%sql
-- Reusable view: the most recent run for each layer.
-- ---------------------------------------------------------------------
-- 5. DQ DASHBOARD QUERIES  (slide 56: health in 10 seconds)
--    Each block is one tile. All filter to the latest run per layer, so
--    the dashboard shows current state while the table keeps history.
-- ---------------------------------------------------------------------
CREATE OR REPLACE VIEW `ftw-week-07`.`01-raw-dev`.v_dq_latest AS
WITH runs AS (
    SELECT layer, run_id, MAX(executed_at) AS run_at
    FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
    GROUP BY layer, run_id
),
latest AS (
    SELECT layer, run_id,
           DENSE_RANK() OVER (PARTITION BY layer ORDER BY run_at DESC) AS rnk
    FROM runs
)
SELECT r.*
FROM `ftw-week-07`.`01-raw-dev`.dq_check_results r
JOIN latest l ON r.layer = l.layer AND r.run_id = l.run_id
WHERE l.rnk = 1;

In [0]:
%sql
-- TILE 1 — Overall health + pass rate + last checked
SELECT
    layer,
    COUNT(*)                                                  AS checks_run,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)          AS passed,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END)          AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END)          AS failed,
    ROUND(100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END), 0), 1) AS pass_rate_pct,
    CASE WHEN SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) > 0 THEN 'STOP'
         WHEN SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) > 0 THEN 'REVIEW'
         ELSE 'HEALTHY' END                                   AS overall_health,
    MAX(executed_at)                                          AS last_checked
FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest
GROUP BY layer;

layer,checks_run,passed,warnings,failed,pass_rate_pct,overall_health,last_checked
raw,73,60,5,0,92.3,REVIEW,2026-09-09T03:26:21.411Z
raw-orig,73,60,5,0,92.3,REVIEW,2026-09-09T03:29:13.196Z


In [0]:
%sql
-- TILE 2 — Failures by dataset (where is the problem?)
SELECT layer, dataset,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
       SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
       SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed
FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest
WHERE status <> 'INFO'
GROUP BY layer, dataset
ORDER BY failed DESC, warnings DESC, dataset;

layer,dataset,failed,warnings,passed
raw,student_registration,0,2,7
raw-orig,student_registration,0,2,7
raw,assessments,0,1,9
raw-orig,assessments,0,1,9
raw,cross_table,0,1,3
raw-orig,cross_table,0,1,3
raw-orig,student_info,0,1,12
raw,student_info,0,1,12
raw-orig,courses,0,0,5
raw,courses,0,0,5


In [0]:
%sql
-- TILE 3 — Failures by check type (what kind of problem?)
SELECT check_type,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
       SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
       COUNT(*)                                         AS total_checks
FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest
WHERE status <> 'INFO'
GROUP BY check_type
ORDER BY failed DESC, warnings DESC;

check_type,failed,warnings,total_checks
NOT_NULL,0,4,18
CONSISTENCY,0,4,10
DOMAIN,0,2,20
VOLUME,0,0,32
REFERENTIAL,0,0,18
RANGE,0,0,10
FORMAT,0,0,10
UNIQUE,0,0,12


In [0]:
%sql
-- TILE 4 — Open issues, worst first (what is broken, and how badly?)
SELECT layer, dataset, check_name, check_type, status,
       fail_count, total_count,
       ROUND(fail_pct * 100, 4)      AS fail_pct,
       ROUND(threshold_pct * 100, 4) AS threshold_pct,
       owner, details
FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest
WHERE status IN ('FAIL', 'WARN')
ORDER BY CASE status WHEN 'FAIL' THEN 0 ELSE 1 END, fail_pct DESC NULLS LAST;

layer,dataset,check_name,check_type,status,fail_count,total_count,fail_pct,threshold_pct,owner,details
raw-orig,student_info,domain_imd_band_format,DOMAIN,WARN,3516,32593,10.7876,0.0,data-engineering,Bands outside the canonical set. `10-20` ships without the % suffix in the source.
raw,student_info,domain_imd_band_format,DOMAIN,WARN,3516,32593,10.7876,0.0,data-engineering,Bands outside the canonical set. `10-20` ships without the % suffix in the source.
raw,assessments,weight_sums_per_presentation,CONSISTENCY,WARN,5,206,2.4272,0.0,data-engineering,Presentations where TMA+CMA <> 100 or Exam <> 100. Blocks any weighted-score measure.
raw-orig,assessments,weight_sums_per_presentation,CONSISTENCY,WARN,5,206,2.4272,0.0,data-engineering,Presentations where TMA+CMA <> 100 or Exam <> 100. Blocks any weighted-score measure.
raw-orig,cross_table,withdrawal_flag_agrees_with_date,CONSISTENCY,WARN,102,32593,0.313,0.0,data-engineering,final_result and unregistration date disagree. Pick one as authoritative before Mart.
raw,cross_table,withdrawal_flag_agrees_with_date,CONSISTENCY,WARN,102,32593,0.313,0.0,data-engineering,final_result and unregistration date disagree. Pick one as authoritative before Mart.
raw-orig,student_registration,completeness_date_registration,NOT_NULL,WARN,45,32593,0.1381,0.5,data-engineering,"Known gap — see decisions.md. Clean stores NULL, not 0. FAILs above 0.5% of rows."
raw,student_registration,completeness_date_registration,NOT_NULL,WARN,45,32593,0.1381,0.5,data-engineering,"Known gap — see decisions.md. Clean stores NULL, not 0. FAILs above 0.5% of rows."
raw-orig,student_registration,completeness_both_dates_missing,NOT_NULL,WARN,6,32593,0.0184,0.1,data-engineering,No derivable timeline. Flag in Clean so Mart excludes them from duration metrics.
raw,student_registration,completeness_both_dates_missing,NOT_NULL,WARN,6,32593,0.0184,0.1,data-engineering,No derivable timeline. Flag in Clean so Mart excludes them from duration metrics.


In [0]:
%sql
-- TILE 5 — Measurements (numbers Clean and Mart must reconcile against)
SELECT layer, dataset, check_name, metric_value, details
FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest
WHERE check_type IN ('MEASURE', 'RECONCILIATION')
ORDER BY dataset, check_name;

layer,dataset,check_name,metric_value,details
raw,assessments,measure_missing_due_date,11.0,Assessments with no due day. Expected: Exam rows only.
raw-orig,assessments,measure_missing_due_date,11.0,Assessments with no due day. Expected: Exam rows only.
raw,student_assessment,measure_banked_scores,1909.0,"Scores transferred from an earlier presentation, not earned in this one."
raw-orig,student_assessment,measure_banked_scores,1909.0,"Scores transferred from an earlier presentation, not earned in this one."
raw,student_assessment,measure_missing_score,173.0,"Submissions with no score. Clean must use NULL, not 0 — different facts."
raw-orig,student_assessment,measure_missing_score,173.0,"Submissions with no score. Clean must use NULL, not 0 — different facts."
raw-orig,student_info,measure_distinct_students,28785.0,Distinct students vs enrollments. The gap is students taking multiple presentations.
raw,student_info,measure_distinct_students,28785.0,Distinct students vs enrollments. The gap is students taking multiple presentations.
raw-orig,student_info,measure_missing_imd_band,1111.0,"Enrollments with no IMD band. Clean must store NULL or ""?"" becomes a category."
raw,student_info,measure_missing_imd_band,1111.0,"Enrollments with no IMD band. Clean must store NULL or ""?"" becomes a category."


In [0]:
%sql
-- TILE 6 — Trend: is quality getting worse?
SELECT
    run_id,
    layer,
    MIN(executed_at) AS run_at,
    SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END) AS checks_scored,
    ROUND(100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END), 0), 1) AS pass_rate_pct,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed
FROM `ftw-week-07`.`01-raw-dev`.dq_check_results
GROUP BY run_id, layer
ORDER BY run_at DESC;

run_id,layer,run_at,checks_scored,pass_rate_pct,warnings,failed
1ad62aa9-6301-4c49-989b-fe1113cca3bf,raw-orig,2026-09-09T03:28:25.447Z,65,92.3,5,0
77446e51-2c81-4884-b4fd-874d1f714266,raw,2026-09-09T03:25:28.748Z,65,92.3,5,0
3df6e952-ecf3-4b89-bfd2-c84315a63e34,raw-orig,2026-09-09T03:20:59.516Z,57,91.2,5,0
f3912ae9-e890-4734-9f75-831144b455f4,raw,2026-09-09T03:18:54.703Z,65,92.3,5,0
68797447-c3b4-4a1a-91e2-a68a313dc793,raw,2026-09-09T02:40:51.139Z,65,92.3,5,0
5d9dd76b-7707-40b6-88e7-623dff745b38,raw,2026-09-09T02:31:36.379Z,65,92.3,5,0
c5bd638a-f77a-4683-a485-5249f751e0d9,raw,2026-09-09T02:24:30.841Z,57,91.2,5,0
79f9474d-af91-40ef-a387-4b4ffe329204,raw,2026-09-08T15:55:06.557Z,57,91.2,5,0
ce525765-5d71-49bf-aac8-a5316fafad6c,raw,2026-09-08T15:34:25.351Z,49,89.8,5,0


In [0]:
%sql
WITH probe AS (
    SELECT CAST(module_presentation_length AS STRING) AS module_presentation_length
    FROM `ftw-week-07`.`01-raw`.courses
    UNION ALL SELECT '0'
    UNION ALL SELECT '?'
    UNION ALL SELECT 'abc'
),
scored AS (
    SELECT COUNT(*) AS total_count,
           SUM(CASE WHEN TRY_CAST(module_presentation_length AS INT) IS NULL
                      OR TRY_CAST(module_presentation_length AS INT) <= 0
                    THEN 1 ELSE 0 END) AS fail_count
    FROM probe
)
SELECT fail_count, total_count,
       CASE WHEN fail_count = 0 THEN 'PASS (check is broken)' ELSE 'FAIL (check works)' END AS verdict
FROM scored;

fail_count,total_count,verdict
3,25,FAIL (check works)


## 6 — Comparison

Both builds hold the same seven source files, so every non-VOLUME check should
return `MATCH`. A `DIFFER` means the two ingestions produced different **data** —
stop and investigate rather than picking a winner.

The 8 `volume_stable_vs_previous_run` rows will differ on the first run of this
notebook: `raw-orig` is a new layer with no previous run, so those record as
`INFO` with a null baseline. Run this notebook twice if you want them scoreable.

What this proves: the two builds hold the same data today. What it cannot prove:
re-run behaviour, atomicity, or row provenance — none of those are properties of
data at rest, so no check can see them.

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 6. COMPARE: Bri's 01-raw vs the 01-raw-dev rebuild
--    FULL OUTER JOIN so a check present in one layer and missing from the
--    other surfaces as a row instead of silently disappearing.
--    Run the dev notebook (layer = 'raw') before this.
-- ---------------------------------------------------------------------
WITH orig AS (
    SELECT * FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest WHERE layer = 'raw-orig'
),
dev AS (
    SELECT * FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest WHERE layer = 'raw'
)
SELECT
    COALESCE(o.dataset, d.dataset)       AS dataset,
    COALESCE(o.check_name, d.check_name) AS check_name,
    o.fail_count  AS orig_fail,  d.fail_count  AS dev_fail,
    o.total_count AS orig_total, d.total_count AS dev_total,
    o.status      AS orig_status, d.status     AS dev_status,
    CASE WHEN o.check_name IS NULL THEN 'DEV ONLY'
         WHEN d.check_name IS NULL THEN 'ORIG ONLY'
         WHEN o.fail_count = d.fail_count
          AND o.total_count = d.total_count THEN 'MATCH'
         ELSE 'DIFFER' END AS verdict
FROM orig o
FULL OUTER JOIN dev d
  ON o.dataset = d.dataset AND o.check_name = d.check_name
ORDER BY CASE WHEN o.check_name IS NULL THEN 2
              WHEN d.check_name IS NULL THEN 1
              WHEN o.fail_count = d.fail_count AND o.total_count = d.total_count THEN 3
              ELSE 0 END,
         dataset, check_name;

dataset,check_name,orig_fail,dev_fail,orig_total,dev_total,orig_status,dev_status,verdict
assessments,domain_assessment_type,0,0,206,206,PASS,PASS,MATCH
assessments,fk_to_courses,0,0,206,206,PASS,PASS,MATCH
assessments,format_date_numeric,0,0,206,206,PASS,PASS,MATCH
assessments,measure_missing_due_date,0,0,206,206,INFO,INFO,MATCH
assessments,missing_date_only_on_exams,0,0,206,206,PASS,PASS,MATCH
assessments,not_null_structural,0,0,206,206,PASS,PASS,MATCH
assessments,range_weight_0_100,0,0,206,206,PASS,PASS,MATCH
assessments,row_count_not_empty,0,0,206,206,PASS,PASS,MATCH
assessments,unique_id_assessment,0,0,206,206,PASS,PASS,MATCH
assessments,volume_stable_vs_previous_run,0,0,206,206,PASS,PASS,MATCH


In [0]:
%sql
-- One-line verdict. Expect every non-VOLUME check to MATCH.
WITH orig AS (SELECT * FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest WHERE layer = 'raw-orig'),
dev  AS (SELECT * FROM `ftw-week-07`.`01-raw-dev`.v_dq_latest WHERE layer = 'raw'),
j AS (
    SELECT o.check_name AS o_name, d.check_name AS d_name,
           o.fail_count AS o_f, d.fail_count AS d_f,
           o.total_count AS o_t, d.total_count AS d_t,
           COALESCE(o.check_type, d.check_type) AS check_type
    FROM orig o FULL OUTER JOIN dev d
      ON o.dataset = d.dataset AND o.check_name = d.check_name
)
SELECT
    COUNT(*)                                                             AS checks_compared,
    SUM(CASE WHEN o_name IS NOT NULL AND d_name IS NOT NULL
              AND o_f = d_f AND o_t = d_t THEN 1 ELSE 0 END)             AS matched,
    SUM(CASE WHEN o_name IS NOT NULL AND d_name IS NOT NULL
              AND (o_f <> d_f OR o_t <> d_t) THEN 1 ELSE 0 END)          AS differing,
    SUM(CASE WHEN o_name IS NULL OR d_name IS NULL THEN 1 ELSE 0 END)    AS unmatched,
    SUM(CASE WHEN check_type = 'VOLUME' AND (o_f <> d_f OR o_t <> d_t)
             THEN 1 ELSE 0 END)                                          AS differing_volume_expected
FROM j;

checks_compared,matched,differing,unmatched,differing_volume_expected
73,73,0,0,0
